In [1]:
# Colab-Setup — wird in lokalen Umgebungen automatisch übersprungen.
import sys

if 'google.colab' in sys.modules:
    import os

    REPO_DIR = '/content/Wassermengenwirtschaft_und_Klimawandel'

    # 1) Repository klonen (idempotent)
    if not os.path.isdir(REPO_DIR):
        print('Colab-Setup: Repository wird geklont ...')
        get_ipython().system('git clone https://github.com/gjohnen1/Wassermengenwirtschaft_und_Klimawandel.git ' + REPO_DIR)
    get_ipython().run_line_magic('cd', REPO_DIR)

    # 2) Nur die RTC-Pakete installieren — nicht requirements.txt als Ganzes,
    #    um numpy==1.26.0 (Hausarbeit/pysheds) nicht auf Colab zu erzwingen.
    print('Colab-Setup: Installiere RTC-Pakete ...')
    get_ipython().run_line_magic(
        'pip',
        'install -q rtc-tools rtc-tools-channel-flow rtc-tools-interface'
    )

    # 3) ipywidgets-Rendering in Colab aktivieren (sonst statisch).
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()

    print('Colab-Setup fertig.')
else:
    print('Nicht in Colab erkannt — lokaler/Binder-Modus wird genutzt.')


Nicht in Colab erkannt — lokaler/Binder-Modus wird genutzt.


In [2]:
# Smoke-Check: läuft RTC-Tools im Kernel?
import sys, platform, importlib

print(f"Plattform:  {platform.system()} ({platform.machine()})")
print(f"Python:     {sys.version.split()[0]}")

missing = []
for pkg in ("rtctools", "casadi", "pymoca", "rtctools_interface",
            "pandas", "plotly", "ipywidgets"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    raise ImportError(
        "Fehlende Pakete: " + ", ".join(missing) +
        ". Bitte in der aktiven Conda-Umgebung (z.B. 'wbw') "
        "'pip install -r requirements.txt' ausführen."
    )

import rtctools, casadi
print(f"rtctools:   {rtctools.__version__}")
print(f"casadi:     {casadi.__version__}")
print("Alle Kernel-Pakete vorhanden — bereit für die Optimierung.")


Plattform:  Windows (AMD64)
Python:     3.10.16


rtctools:   2.7.3
casadi:     3.7.2
Alle Kernel-Pakete vorhanden — bereit für die Optimierung.


In [3]:
import os
import sys
import shutil
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    """Repo-Wurzel finden: enthält 'Inhalt/simple_reservoir/BlueRiver2'."""
    for p in [start, *start.parents]:
        if (p / "Inhalt" / "simple_reservoir" / "BlueRiver2").exists():
            return p
    raise FileNotFoundError(
        "Repository-Wurzel mit Inhalt/simple_reservoir/BlueRiver2 nicht gefunden."
    )

REPO_ROOT     = find_repo_root(Path.cwd())
BLUERIVER_DIR = REPO_ROOT / "Inhalt" / "simple_reservoir" / "BlueRiver2"
DATA_ROOT     = REPO_ROOT / "Inhalt" / "Notebook_Daten" / "RTC_BlueRiver"

SCENARIO_SHORT = DATA_ROOT / "ShortTerm" / "input"
SCENARIO_LONG  = DATA_ROOT / "LongTerm"  / "input"
DATA_MODEL     = DATA_ROOT / "model"

CASE_DIR    = DATA_ROOT / "runtime_case"
CASE_INPUT  = CASE_DIR / "input"
CASE_MODEL  = CASE_DIR / "model"
CASE_SRC    = CASE_DIR / "src"
CASE_XSD    = CASE_DIR / "xsd"
CASE_OUTPUT = CASE_DIR / "output"

# Statt einer separaten RTC-Conda-Sub-Umgebung läuft das Modell direkt im
# Kernel-Interpreter der aktiven Conda-Umgebung (z.B. 'wbw').
PY = sys.executable

print(f"Repo root:     {REPO_ROOT}")
print(f"BlueRiver:     {BLUERIVER_DIR}")
print(f"Data root:     {DATA_ROOT}")
print(f"Runtime case:  {CASE_DIR}")
print(f"Python:        {PY}")


Repo root:     C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel
BlueRiver:     C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\simple_reservoir\BlueRiver2
Data root:     C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver
Runtime case:  C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case
Python:        C:\Users\grego\anaconda3\envs\hausarbeit_wb4\python.exe


In [4]:
required_blue = [
    BLUERIVER_DIR / "src"   / "BlueRiver.py",
    BLUERIVER_DIR / "model" / "BlueRiver.mo",
    BLUERIVER_DIR / "xsd"   / "rtcDataConfig.xsd",
    BLUERIVER_DIR / "xsd"   / "rtcSharedTypes.xsd",
]

required_data = [
    DATA_MODEL / "reservoirs.csv",
    DATA_MODEL / "volumelevel.csv",
]

required_short = [
    SCENARIO_SHORT / "goal_table.csv",
    SCENARIO_SHORT / "plot_table.csv",
    SCENARIO_SHORT / "rtcDataConfig.xml",
    SCENARIO_SHORT / "rtcParameterConfig.xml",
    SCENARIO_SHORT / "timeseries_import.csv",
    SCENARIO_SHORT / "timeseries_import.xml",
]

missing = [p for p in (required_blue + required_data + required_short) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Folgende Dateien fehlen:\n" + "\n".join(str(p) for p in missing)
    )

print("Alle Pflichtdateien vorhanden.")
print(f"BlueRiver.py:  {required_blue[0]}")
print(f"BlueRiver.mo:  {required_blue[1]}")


Alle Pflichtdateien vorhanden.
BlueRiver.py:  C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\simple_reservoir\BlueRiver2\src\BlueRiver.py
BlueRiver.mo:  C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\simple_reservoir\BlueRiver2\model\BlueRiver.mo


In [5]:
import shutil
import stat
import time

def _on_rm_error(func, path, exc_info):
    # Windows/OneDrive: read-only Attribute entfernen und erneut versuchen
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception:
        pass

def _remove_path(path: Path):
    if path.is_dir() and not path.is_symlink():
        shutil.rmtree(path, onerror=_on_rm_error)
    else:
        try:
            os.chmod(path, stat.S_IWRITE)
        except Exception:
            pass
        path.unlink(missing_ok=True)

def _clear_dir(path: Path, retries: int = 5, delay_s: float = 0.4):
    path.mkdir(parents=True, exist_ok=True)
    for child in list(path.iterdir()):
        for attempt in range(1, retries + 1):
            try:
                _remove_path(child)
                break
            except PermissionError:
                if attempt == retries:
                    raise
                time.sleep(delay_s * attempt)

def prepare_case(scenario_input: Path):
    # Frische Ordnerstruktur (Output bleibt erhalten, damit figures/<RUN_TAG>/
    # zwischen Konfigurationen nicht überschrieben werden).
    for p in [CASE_INPUT, CASE_MODEL, CASE_SRC, CASE_XSD]:
        _clear_dir(p)
    CASE_OUTPUT.mkdir(parents=True, exist_ok=True)

    shutil.copytree(BLUERIVER_DIR / "src",   CASE_SRC,   dirs_exist_ok=True)
    shutil.copytree(BLUERIVER_DIR / "model", CASE_MODEL, dirs_exist_ok=True)
    shutil.copytree(BLUERIVER_DIR / "xsd",   CASE_XSD,   dirs_exist_ok=True)

    shutil.copy2(DATA_MODEL / "reservoirs.csv",  CASE_MODEL / "reservoirs.csv")
    shutil.copy2(DATA_MODEL / "volumelevel.csv", CASE_MODEL / "volumelevel.csv")

    shutil.copytree(scenario_input, CASE_INPUT, dirs_exist_ok=True)

    print("Case vorbereitet:", CASE_DIR)
    print("Input:",  CASE_INPUT)
    print("Output:", CASE_OUTPUT)


In [6]:
prepare_case(SCENARIO_SHORT)
print("ShortTerm-Szenario ist aktiv.")


Case vorbereitet: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case
Input: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\input
Output: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\output
ShortTerm-Szenario ist aktiv.


In [7]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

GOAL_TABLE_RUNTIME = CASE_INPUT / "goal_table.csv"

if not GOAL_TABLE_RUNTIME.exists():
    raise FileNotFoundError(f"Fehlende Datei: {GOAL_TABLE_RUNTIME}")

_editor_df = pd.read_csv(GOAL_TABLE_RUNTIME)
_row_controls = {}


def _bool_from_cell(v):
    if pd.isna(v):
        return False
    if isinstance(v, str):
        return v.strip().lower() in {"1", "true", "yes", "y"}
    return bool(int(v)) if isinstance(v, (int, float)) else bool(v)


def _int_or_default(v, default=0):
    try:
        return int(float(v))
    except Exception:
        return default


def _float_or_default(v, default=1.0):
    try:
        return float(v)
    except Exception:
        return default


def _build_row(idx, row):
    id_val = _int_or_default(row.get("id"), idx)
    state = str(row.get("state", ""))
    goal_type = str(row.get("goal_type", ""))

    meta = widgets.HTML(value=f"<b>ID {id_val}</b> | {state} | {goal_type}",
                        layout=widgets.Layout(width="360px"))
    active = widgets.Checkbox(value=_bool_from_cell(row.get("active", 0)),
                              description="active", indent=False,
                              layout=widgets.Layout(width="90px"))
    priority = widgets.BoundedIntText(value=_int_or_default(row.get("priority"), 1),
                                      min=0, max=100, description="prio",
                                      layout=widgets.Layout(width="140px"))
    weight = widgets.FloatText(value=_float_or_default(row.get("weight"), 1.0),
                               description="weight",
                               layout=widgets.Layout(width="160px"))
    order = widgets.BoundedIntText(value=max(1, _int_or_default(row.get("order"), 1)),
                                   min=1, max=10, description="order",
                                   layout=widgets.Layout(width="150px"))

    box = widgets.HBox([meta, active, priority, weight, order])
    _row_controls[idx] = {"active": active, "priority": priority,
                          "weight": weight, "order": order}
    return box


def _collect_df_from_controls(base_df):
    df = base_df.copy()
    for idx in df.index:
        c = _row_controls[idx]
        df.at[idx, "active"]   = 1 if c["active"].value else 0
        df.at[idx, "priority"] = int(c["priority"].value)
        df.at[idx, "weight"]   = float(c["weight"].value)
        df.at[idx, "order"]    = int(c["order"].value)
    return df


status = widgets.Output(layout=widgets.Layout(border="1px solid #ddd",
                                              padding="6px",
                                              max_height="160px",
                                              overflow="auto"))

save_btn = widgets.Button(description="Goals speichern", button_style="success",
                          icon="save",
                          layout=widgets.Layout(width="220px"))


def _on_save(_):
    base_df = pd.read_csv(GOAL_TABLE_RUNTIME)
    new_df = _collect_df_from_controls(base_df)
    new_df.to_csv(GOAL_TABLE_RUNTIME, index=False)
    with status:
        clear_output(wait=True)
        print(f"Gespeichert nach: {GOAL_TABLE_RUNTIME}")
        print()
        print(new_df[["id", "state", "active", "priority", "weight", "order"]].to_string(index=False))


save_btn.on_click(_on_save)

rows = [_build_row(idx, row) for idx, row in _editor_df.iterrows()]

display(widgets.VBox([
    widgets.HTML(
        "<b>Goal-Editor</b> — Werte anpassen und auf <i>Goals speichern</i> klicken. "
        f"Die Datei wird unter <code>{GOAL_TABLE_RUNTIME}</code> abgelegt."
    ),
    widgets.VBox(rows),
    save_btn,
    status,
]))

with status:
    clear_output(wait=True)
    print(f"Editor bereit. Aktuelle Datei: {GOAL_TABLE_RUNTIME}")


In [8]:
# In-Process-Lauf: das BlueRiver-Modell läuft direkt im Jupyter-Kernel.
# Subprozess-Alternative lädt RTC-Tools/CasADi/IPOPT doppelt in den Speicher
# und sprengt auf Binder (2 GB) das Limit → OOM-Kill → "File Save Error 424".
# RTC-Tools-Log wird in eine Datei umgeleitet, damit die Zelle nicht Megabytes
# an Solver-Output einsammelt (zweite Ursache für den 424-Fehler).

import importlib, logging, sys
from contextlib import redirect_stdout, redirect_stderr


def _run_rtc(log_path):
    """Run BlueRiver in-process. Returncode: 0 = OK, !=0 = Fehler."""
    if str(CASE_SRC) not in sys.path:
        sys.path.insert(0, str(CASE_SRC))
    sys.modules.pop("BlueRiver", None)   # prepare_case-Änderungen übernehmen
    prev_cwd = os.getcwd()
    os.chdir(CASE_SRC)

    rtc_log = logging.getLogger("rtctools")
    fh = logging.FileHandler(log_path, mode="w", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s: %(message)s"))
    prev_h, prev_p = rtc_log.handlers[:], rtc_log.propagate
    rtc_log.handlers, rtc_log.propagate = [fh], False
    rtc_log.setLevel(logging.INFO)

    rc = 0
    try:
        with open(log_path, "a", encoding="utf-8") as lf, \
             redirect_stdout(lf), redirect_stderr(lf):
            from rtctools.util import run_optimization_problem
            BlueRiver = importlib.import_module("BlueRiver").BlueRiver
            run_optimization_problem(BlueRiver, log_level=logging.INFO,
                                     plotting_library="matplotlib")
    except SystemExit as exc:
        rc = int(exc.code or 0)
    except Exception:
        import traceback
        with open(log_path, "a", encoding="utf-8") as lf:
            traceback.print_exc(file=lf)
        rc = 1
    finally:
        os.chdir(prev_cwd)
        rtc_log.handlers, rtc_log.propagate = prev_h, prev_p
        fh.close()
    return rc


RUN_TAG = "shortterm_default"
print(f"RUN_TAG: {RUN_TAG}")

log_path = CASE_DIR / "run-log.txt"
returncode = _run_rtc(log_path)

print(f"Return code: {returncode}")
print(f"Log:         {log_path}")
print("--- Letzte Logzeilen ---")
print("".join(open(log_path, encoding="utf-8").readlines()[-10:]))

if returncode != 0:
    raise RuntimeError("BlueRiver-Lauf fehlgeschlagen. Siehe run-log.txt.")


RUN_TAG: shortterm_default


Return code: 0
Log:         C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\run-log.txt
--- Letzte Logzeilen ---
Total seconds in IPOPT                               = 0.044

EXIT: Optimal Solution Found.
         nlp  :   t_proc      (avg)   t_wall      (avg)    n_eval
       nlp_f  |        0 (       0) 973.00us ( 31.39us)        31
       nlp_g  |        0 (       0)   1.08ms ( 34.90us)        31
  nlp_grad_f  |   6.00ms (120.00us)   4.13ms ( 82.50us)        50
  nlp_hess_l  |        0 (       0)   7.00us (304.35ns)        23
   nlp_jac_g  |   8.00ms (320.00us)   3.10ms (124.08us)        25
       total  |  49.00ms ( 49.00ms)  44.16ms ( 44.16ms)         1



In [9]:
import xml.etree.ElementTree as ET
import pandas as pd

out_csv = CASE_OUTPUT / "timeseries_export.csv"
out_diag = CASE_OUTPUT / "diag.xml"
perf_dir = CASE_OUTPUT / "performance_metrics"
run_log = CASE_DIR / "run-log.txt"

if not out_csv.exists():
    raise FileNotFoundError(f"Fehlt: {out_csv}")

df = pd.read_csv(out_csv, parse_dates=["time"])
required_cols = ["time", "TroutLake_V", "TroutLake_Q_out", "RiverCity_Q"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError("Fehlende Spalten in timeseries_export.csv: " + ", ".join(missing_cols))
if df.empty:
    raise ValueError("timeseries_export.csv ist leer.")

diag_info = "Keine Diag-Information gefunden."
if out_diag.exists():
    root = ET.parse(out_diag).getroot()
    lines = list(root)
    levels = {"1": 0, "2": 0, "3": 0}
    for line in lines:
        lvl = line.attrib.get("level")
        if lvl in levels:
            levels[lvl] += 1
    diag_info = f"diag.xml vorhanden. Level-Count: {levels}"
elif perf_dir.exists() and list(perf_dir.glob("*.csv")):
    diag_info = f"diag.xml fehlt, aber performance_metrics vorhanden ({len(list(perf_dir.glob('*.csv')))} Dateien)."
elif run_log.exists():
    log_text = run_log.read_text(encoding="utf-8", errors="ignore")
    if "Traceback" in log_text:
        raise RuntimeError("run-log.txt enthaelt einen Traceback. Lauf pruefen.")
    if "Done goal programming" in log_text:
        diag_info = "diag.xml fehlt, aber Lauf laut run-log.txt erfolgreich (Done goal programming)."
    else:
        raise FileNotFoundError("Weder diag.xml noch performance_metrics gefunden. Laufpruefung unvollstaendig.")
else:
    raise FileNotFoundError("Weder diag.xml noch performance_metrics/run-log.txt gefunden.")

print("CSV ok. Zeilen:", len(df))
print(diag_info)
short_term_df = df.copy()


CSV ok. Zeilen: 57
diag.xml fehlt, aber performance_metrics vorhanden (4 Dateien).


In [10]:
# Plotly-Renderer auf CDN-basiert setzen — sonst landet plotly.js (~3 MB)
# pro Figur INLINE in der gerenderten HTML-Seite (Jupyter-Book-Build).
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import json
from collections import OrderedDict

import pandas as pd
import plotly.graph_objects as go


CACHE_DIR = CASE_OUTPUT / "cached_results"
cache_files = sorted(CACHE_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime)
if not cache_files:
    raise FileNotFoundError(f"Keine cached_results-Datei gefunden in: {CACHE_DIR}")

cache_path = cache_files[-1]
cache_data = json.loads(cache_path.read_text(encoding="utf-8"))

intermediate_results = cache_data.get("intermediate_results", [])
if not intermediate_results:
    raise ValueError("cached_results enthält keine intermediate_results.")


def _unwrap_array(value):
    if isinstance(value, dict) and "data" in value:
        return value["data"]
    return value


io_datetimes = cache_data["prio_independent_data"]["io_datetimes"]
time_index = pd.to_datetime([item["value"] for item in io_datetimes])

base_goals = {
    int(goal["goal_id"]): goal
    for goal in cache_data["prio_independent_data"]["base_goals"]
}
plot_rows = [item["value"] for item in cache_data["plot_options"]["plot_config"]]

color_map = {
    "TroutLake_V":         "royalblue",
    "RiverCity_Q":         "green",
    "TroutLake_Q_out":     "pink",
    "Alder_Inflow":        "olive",
    "TroutLake_Q_spill":   "brown",
    "TroutLake_Q_turbine": "violet",
}


def _add_target(fig, name, target_series):
    if target_series is None:
        return
    values = _unwrap_array(target_series)
    if not values:
        return
    numeric = [float(v) for v in values]
    if all(abs(v - numeric[0]) < 1e-12 for v in numeric):
        fig.add_hline(
            y=numeric[0], line_dash="dash", line_color="red",
            annotation_text=name,
            annotation_position="top right" if "max" in name.lower() else "bottom right",
        )
    else:
        fig.add_trace(go.Scatter(
            x=time_index[: len(numeric)], y=numeric, mode="lines",
            name=name, line=dict(color="red", dash="dash"),
        ))


def _build_figures_for_priority(priority, snapshot):
    """Return an OrderedDict[stem -> Figure] for a single priority snapshot."""
    timeseries_data = {
        key: _unwrap_array(value)
        for key, value in snapshot.get("timeseries_data", {}).items()
    }
    figs = OrderedDict()
    for row in plot_rows:
        goal_id = int(row["id"])
        goal = base_goals.get(goal_id)
        if goal is None:
            continue

        main_var = goal.get("state")
        extra_vars = row.get("variables_style_2", [])
        variables = [main_var] + [v for v in extra_vars if v != main_var]

        fig = go.Figure()
        for var in variables:
            values = timeseries_data.get(var)
            if values is None:
                continue
            y = [float(v) for v in values]
            x = time_index[: len(y)]
            fig.add_trace(go.Scatter(
                x=x, y=y, mode="lines", name=var,
                line=dict(color=color_map.get(var, None)),
            ))

        _add_target(fig, "Target max", goal.get("target_max_series"))
        _add_target(fig, "Target min", goal.get("target_min_series"))

        subtitle = row.get("custom_title", f"Goal {goal_id}")
        title = f"Priority {priority} | {subtitle}"
        fig.update_layout(
            title=title, xaxis_title="Time",
            yaxis_title=str(row.get("y_axis_title", "Value")).replace("$", ""),
            template="plotly_white",
        )

        # Stable filename stem
        safe_state = str(main_var or "var").replace("/", "_").replace(" ", "_")
        figs[f"priority_{priority:02d}__goal_{goal_id:03d}__{safe_state}"] = fig
    return figs


available_priorities = sorted(int(item.get("priority")) for item in intermediate_results)
figures = OrderedDict()
for prio in available_priorities:
    snapshot = next(item for item in intermediate_results if int(item.get("priority")) == prio)
    figures.update(_build_figures_for_priority(prio, snapshot))

print(f"Cached results: {cache_path.name}")
print(f"Verfügbare Prioritäten: {available_priorities}")
print(f"Anzahl erzeugter Figuren: {len(figures)}")

# Inline: nur die erste Figur anzeigen — alle Figuren werden in der
# Export-Zelle darunter als HTML/PNG geschrieben. Spart Notebook-
# Speicher (Binder/JupyterHub brechen Saves oberhalb ~10 MB mit 424 ab).
next(iter(figures.values())).show()


Cached results: 1776760193.json
Verfügbare Prioritäten: [1, 3, 4, 9]
Anzahl erzeugter Figuren: 16


In [11]:
import plotly.graph_objects as go

RUN_LONGTERM = True

if not RUN_LONGTERM:
    print("LongTerm übersprungen. RUN_LONGTERM=True zum Aktivieren.")
else:
    prepare_case(SCENARIO_LONG)
    long_log = CASE_DIR / "run-log.txt"
    rc_long = _run_rtc(long_log)
    if rc_long != 0:
        print("".join(open(long_log, encoding="utf-8").readlines()[-10:]))
        raise RuntimeError("LongTerm-Lauf fehlgeschlagen. Siehe run-log.txt.")

    long_df = pd.read_csv(CASE_OUTPUT / "timeseries_export.csv", parse_dates=["time"])
    print("LongTerm-Zeilen:", len(long_df))

    def _compare_fig(col, title, colors):
        fig = go.Figure([
            go.Scatter(x=short_term_df["time"], y=short_term_df[col],
                       mode="lines", name="ShortTerm", line=dict(color=colors[0])),
            go.Scatter(x=long_df["time"], y=long_df[col],
                       mode="lines", name="LongTerm", line=dict(color=colors[1])),
        ])
        fig.update_layout(title=title, xaxis_title="Time", yaxis_title=col,
                          template="plotly_white")
        return fig

    fig_v = _compare_fig("TroutLake_V",
                         "Vergleich ShortTerm vs LongTerm: TroutLake_V",
                         ("royalblue", "orange"))
    fig_q = _compare_fig("TroutLake_Q_out",
                         "Vergleich ShortTerm vs LongTerm: TroutLake_Q_out",
                         ("green", "red"))
    fig_v.show()
    fig_q.show()

    long_dir = CASE_OUTPUT / "figures" / f"{RUN_TAG}_longterm_compare"
    long_dir.mkdir(parents=True, exist_ok=True)
    fig_v.write_html(str(long_dir / "compare__TroutLake_V.html"),     include_plotlyjs="cdn")
    fig_q.write_html(str(long_dir / "compare__TroutLake_Q_out.html"), include_plotlyjs="cdn")
    print(f"LongTerm-Vergleichsfiguren als HTML: {long_dir}")


Case vorbereitet: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case
Input: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\input
Output: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\output


LongTerm-Zeilen: 61


LongTerm-Vergleichsfiguren als HTML: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\output\figures\shortterm_default_longterm_compare
